# Agentic Researcher: open-weight bulk research

This notebook runs a previously curated `research-queue/v1` with a local
OpenAI-compatible vLLM server and OpenCode. Commercial curators should create
and approve the queue before this notebook is used. Projects and batch state
are stored on Google Drive so an interrupted Colab session can resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/agentic-researcher')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
QUEUE = DRIVE_ROOT / 'research_queue.json'
WORK_ROOT = DRIVE_ROOT / 'projects'
STATE = DRIVE_ROOT / 'batch_state.json'
assert QUEUE.exists(), f'Upload the curated queue to {QUEUE}'


In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = os.environ.get('AGENTIC_RESEARCHER_REPO', 'https://github.com/Erikiss/The-Agentic-Researcher.git')
REPO_REF = os.environ.get('AGENTIC_RESEARCHER_REF', 'main')
repo = Path('/content/The-Agentic-Researcher')
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(repo)], check=True)
!pip -q install uv
# vLLM needs its own environment: Colab's preinstalled torch is built for an
# older CUDA than current vLLM wheels, and a --system install keeps that torch,
# leaving vLLM without the CUDA runtime its binaries link against.
!uv venv /content/vllm-venv
!uv pip install --python /content/vllm-venv/bin/python --torch-backend=auto vllm
!npm install -g opencode-ai


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
props = torch.cuda.get_device_properties(0)
if (props.major, props.minor) < (8, 0):
    raise RuntimeError(
        f'{torch.cuda.get_device_name(0)} (compute capability {props.major}.{props.minor}) '
        'cannot run gpt-oss with vLLM: MXFP4 needs compute capability 8.0 or newer, '
        'so switch the Colab runtime to an A100 (or at least L4) GPU.'
    )
vram_gib = props.total_memory / 1024**3
MODEL = 'openai/gpt-oss-120b' if vram_gib >= 75 else 'openai/gpt-oss-20b'
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB)')
print(f'Model: {MODEL}')


In [ ]:
import json, os
from pathlib import Path

config_dir = Path.home() / '.config' / 'opencode'
config_dir.mkdir(parents=True, exist_ok=True)
config = {
    '$schema': 'https://opencode.ai/config.json',
    'autoupdate': False,
    'provider': {
        'local': {
            'npm': '@ai-sdk/openai-compatible',
            'name': 'Local Agentic Researcher',
            'options': {'baseURL': 'http://127.0.0.1:8000/v1', 'apiKey': 'local'},
            'models': {'default': {'name': MODEL}},
        }
    },
    'model': 'local/default',
    'permission': {
        'external_directory': 'deny',
        'question': 'deny',
        'doom_loop': 'deny',
        'bash': {
            '*': 'allow',
            'git push *': 'deny',
            'git commit *': 'deny',
        },
    },
}
(config_dir / 'opencode.json').write_text(json.dumps(config, indent=2), encoding='utf-8')


In [ ]:
import subprocess, sys, time, urllib.request

log_path = DRIVE_ROOT / 'vllm.log'
log_handle = log_path.open('ab')

def fail(exc_type, message):
    log_lines = log_path.read_text(errors='replace').splitlines()
    print(f'--- last {min(len(log_lines), 60)} lines of {log_path} ---')
    print('\n'.join(log_lines[-60:]))
    raise exc_type(message)

server = subprocess.Popen([
    '/content/vllm-venv/bin/vllm', 'serve', MODEL,
    '--served-model-name', 'default',
    '--host', '127.0.0.1',
    '--port', '8000',
    '--max-model-len', '32768',
    # vLLM's default memory split cannot start gpt-oss-120b on a single 80 GiB
    # GPU; the official gpt-oss recipe prescribes these two values against that.
    '--gpu-memory-utilization', '0.95',
    '--max-num-batched-tokens', '1024',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'openai',
], stdout=log_handle, stderr=subprocess.STDOUT)

deadline = time.monotonic() + 3600  # the first start downloads up to ~63 GiB of weights
while time.monotonic() < deadline:
    if server.poll() is not None:
        fail(RuntimeError, f'vLLM exited with code {server.returncode}; full log: {log_path}')
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2)
        print('vLLM is ready')
        break
    except Exception:
        time.sleep(5)
else:
    server.terminate()
    fail(TimeoutError, f'vLLM did not become ready within 60 minutes; full log: {log_path}')


In [ ]:
import os, subprocess, sys

repo = Path('/content/The-Agentic-Researcher')
env = os.environ.copy()
env['PYTHONPATH'] = str(repo)
command = [
    sys.executable, '-m', 'agentic_researcher', 'run-batch', str(QUEUE),
    '--work-root', str(WORK_ROOT),
    '--state', str(STATE),
    '--provider', 'opencode',
    '--command', 'opencode run --auto {prompt}',
    '--timeout', '7200',
]
print(' '.join(command))
subprocess.run(command, cwd=repo, env=env, check=True)


Re-running the final cell is safe. Completed task IDs are read from
`batch_state.json`; only pending or retryable tasks are executed.